# DQN VANILLA

## Prueba de models.py

In [1]:
import sys
import torch

sys.path.append("../src")

from models import DQN

N_ACCIONES = 6
SEMILLA = 42

torch.manual_seed(SEMILLA)

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

modelo = DQN(n_acciones=N_ACCIONES).to(device)

#simular un batch de dos observaciones preprocesadas
observaciones_prueba = torch.randint(
    low=0,
    high=256,
    size=(2, 4, 84, 84),
    dtype=torch.uint8,
    device=device,
)

with torch.no_grad():
    valores_q = modelo(observaciones_prueba)

numero_parametros = sum(
    parametro.numel()
    for parametro in modelo.parameters()
)

print(f"Dispositivo: {device}")
print(f"Forma de entrada: {observaciones_prueba.shape}")
print(f"Forma de salida: {valores_q.shape}")
print(f"Número de parámetros: {numero_parametros:,}")
print("\nValores Q de prueba:")
print(valores_q)

assert valores_q.shape == (2, N_ACCIONES)
print("\nLa arquitectura produce las dimensiones correctas.")

Dispositivo: mps
Forma de entrada: torch.Size([2, 4, 84, 84])
Forma de salida: torch.Size([2, 6])
Número de parámetros: 1,687,206

Valores Q de prueba:
tensor([[ 0.0458, -0.0209, -0.0371, -0.0343, -0.0102,  0.0209],
        [ 0.0580, -0.0216, -0.0406, -0.0354, -0.0103,  0.0161]],
       device='mps:0')

La arquitectura produce las dimensiones correctas.


## Prueba de replay_buffer.py

In [2]:
from replay_buffer import ReplayBuffer

buffer_prueba = ReplayBuffer(
    capacidad=10,
    forma_observacion=(4, 84, 84),
    seed=SEMILLA,
)

#agregar más experiencias que su capacidad para comprobar que reemplaza correctamente las experiencias antiguas
for paso in range(12):
    observacion = torch.randint(
        0,
        256,
        size=(4, 84, 84),
        dtype=torch.uint8,
    ).numpy()

    siguiente_observacion = torch.randint(
        0,
        256,
        size=(4, 84, 84),
        dtype=torch.uint8,
    ).numpy()

    buffer_prueba.agregar(
        observacion=observacion,
        accion=paso % N_ACCIONES,
        recompensa=float(paso),
        siguiente_observacion=siguiente_observacion,
        finalizado=(paso % 5 == 0),
    )

batch = buffer_prueba.muestrear(
    batch_size=4,
    device=device,
)

(
    observaciones_batch,
    acciones_batch,
    recompensas_batch,
    siguientes_observaciones_batch,
    finalizados_batch,
) = batch

print(f"Tamaño actual del buffer: {len(buffer_prueba)}")
print(f"Capacidad máxima: {buffer_prueba.capacidad}")
print(
    f"Memoria aproximada: "
    f"{buffer_prueba.memoria_aproximada_mb:.2f} MB"
)

print("\nFORMAS DEL BATCH")
print(f"Observaciones: {observaciones_batch.shape}")
print(f"Acciones: {acciones_batch.shape}")
print(f"Recompensas: {recompensas_batch.shape}")
print(
    "Siguientes observaciones: "
    f"{siguientes_observaciones_batch.shape}"
)
print(f"Finalizados: {finalizados_batch.shape}")

print("\nTIPOS DE DATOS")
print(f"Observaciones: {observaciones_batch.dtype}")
print(f"Acciones: {acciones_batch.dtype}")
print(f"Recompensas: {recompensas_batch.dtype}")
print(f"Finalizados: {finalizados_batch.dtype}")

assert len(buffer_prueba) == 10
assert observaciones_batch.shape == (4, 4, 84, 84)
assert acciones_batch.shape == (4,)
assert recompensas_batch.shape == (4,)
assert finalizados_batch.shape == (4,)

print("\nEl replay buffer funciona correctamente.")

Tamaño actual del buffer: 10
Capacidad máxima: 10
Memoria aproximada: 0.54 MB

FORMAS DEL BATCH
Observaciones: torch.Size([4, 4, 84, 84])
Acciones: torch.Size([4])
Recompensas: torch.Size([4])
Siguientes observaciones: torch.Size([4, 4, 84, 84])
Finalizados: torch.Size([4])

TIPOS DE DATOS
Observaciones: torch.uint8
Acciones: torch.int64
Recompensas: torch.float32
Finalizados: torch.bool

El replay buffer funciona correctamente.
